# 🌾 Agricultural Crop Plant Image Classification
## End-to-End Deep Learning & Computer Vision Pipeline using PyTorch and Transfer Learning (ResNet-18)

### Project Overview:
This project builds an automated plant vision system capable of identifying **139 distinct crop plant species** from field images.
Accurate identification helps farmers and agronomists:
- Rapidly diagnose crop varieties in precision farming
- Monitor biodiversity and intercropping fields
- Enable automated weeding and robotic harvesting systems


---
## 1. Setup & Environment Configuration
Import core libraries for deep learning (`torch`, `torchvision`), numerical computing (`numpy`), image manipulation (`PIL`), and visualization (`matplotlib`).


In [ ]:
import os
import random
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch Version: {torch.__version__}")


---
## 2. Dataset Inspection & Exploratory Data Analysis (EDA)
Inspect the dataset structure, count classes, and visualize sample crop images.


In [ ]:
dataset_path = "RGB_224x224"
train_dir = os.path.join(dataset_path, "train")
val_dir = os.path.join(dataset_path, "val")
test_dir = os.path.join(dataset_path, "test")

# Verify directory paths
for split_name, split_path in [("Train", train_dir), ("Validation", val_dir), ("Test", test_dir)]:
    num_classes = len(os.listdir(split_path)) if os.path.exists(split_path) else 0
    print(f"{split_name} split: {num_classes} crop classes found.")

classes = sorted(os.listdir(train_dir))
print(f"\nTotal Unique Crop Categories: {len(classes)}")
print(f"Sample Classes: {random.sample(classes, 6)}")


In [ ]:
def show_sample_images(num_classes=6, samples_per_class=1):
    """Displays sample crop images from different classes."""
    selected_classes = random.sample(classes, num_classes)
    fig, axes = plt.subplots(1, num_classes, figsize=(18, 4))
    
    for idx, cls_name in enumerate(selected_classes):
        cls_folder = os.path.join(train_dir, cls_name)
        img_file = os.listdir(cls_folder)[0]
        img_path = os.path.join(cls_folder, img_file)
        
        img = Image.open(img_path).convert('RGB')
        axes[idx].imshow(img)
        axes[idx].set_title(cls_name.replace(' plant', ''), fontsize=10, fontweight='bold')
        axes[idx].axis('off')
        
    plt.tight_layout()
    plt.show()

show_sample_images(num_classes=6)


---
## 3. Data Preprocessing, Augmentation & DataLoaders
Define image transformations:
- **Training Transformations**: Resizing to 224x224, random horizontal flips, mild random rotation, color jittering for lighting invariance, tensor conversion, and ImageNet normalization.
- **Validation / Test Transformations**: Deterministic resizing and standard ImageNet normalization.


In [ ]:
# ImageNet normalization statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Create datasets
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=eval_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=eval_transform)

print(f"Dataset summary:")
print(f"  Training images:   {len(train_dataset):,}")
print(f"  Validation images: {len(val_dataset):,}")
print(f"  Test images:       {len(test_dataset):,}")


In [ ]:
# Create DataLoaders (num_workers=0 ensures seamless execution across all platforms)
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches per epoch: Train={len(train_loader)}, Val={len(val_loader)}, Test={len(test_loader)}")


---
## 4. Transfer Learning Model Architecture (ResNet-18)
Using **ResNet-18** pre-trained on ImageNet:
1. Load pre-trained weights to leverage robust low-level visual features (edges, textures, leaf contours).
2. Freeze backbone convolutional layers.
3. Replace the final fully connected classification head (`fc`) with a new Linear projection mapping 512 features to **139 crop plant classes**.


In [ ]:
# Device configuration: CUDA GPU -> Apple MPS -> CPU
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Active compute device: {device}")

# Load Pre-trained ResNet-18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze feature extractor parameters
for param in model.parameters():
    param.requires_grad = False

# Adapt classification head for 139 crop categories
num_classes = len(train_dataset.classes)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)
print(f"Model modified successfully for {num_classes} crop classes.")


---
## 5. Loss Function, Optimizer & Training Loop
- **Criterion**: `CrossEntropyLoss` for multi-class classification.
- **Optimizer**: `Adam` with learning rate `0.001` optimizing the classification head parameters.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100.0
    return epoch_loss, epoch_acc

def evaluate(model, loader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            # Corrected: Parentheses placed properly around (predicted == labels)
            correct += (predicted == labels).sum().item()

    val_loss = running_loss / total
    val_acc = (correct / total) * 100.0
    return val_loss, val_acc


---
## 6. Training & Validation Execution
Train the model over multiple epochs, track metrics, and checkpoint the best performing weights.
*(Tip: For quick experimentation on CPU, you can train for 3-5 epochs or on a representative subset)*


In [ ]:
epochs = 3
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("="*65)
print(" Starting Training Loop")
print("="*65)

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch [{epoch+1:02d}/{epochs:02d}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs("models", exist_ok=True)
        torch.save(model.state_dict(), "models/crop_resnet18_best_nb.pth")
        print(f"  --> Saved new best checkpoint with Val Acc: {best_val_acc:.2f}%")

print("="*65)
print(f"Training Complete! Best Validation Accuracy: {best_val_acc:.2f}%")


---
## 7. Model Evaluation on Test Dataset
Evaluate on unseen test images to measure generalization performance.


In [ ]:
test_loss, test_acc = evaluate(model, test_loader)
print("="*40)
print(f" Final Test Accuracy: {test_acc:.2f}%")
print(f" Final Test Loss:     {test_loss:.4f}")
print("="*40)


In [ ]:
def visualize_predictions(model, loader, n=6):
    """Displays test images with their true and model-predicted labels."""
    model.eval()
    class_names = train_dataset.classes
    images_shown = 0

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            for i in range(images.size(0)):
                img = images[i].cpu().permute(1, 2, 0).numpy()
                # De-normalize ImageNet stats for display
                img = np.clip(img * IMAGENET_STD + IMAGENET_MEAN, 0, 1)

                true_label = class_names[labels[i]].replace(' plant', '')
                pred_label = class_names[preds[i]].replace(' plant', '')
                is_correct = (labels[i] == preds[i])
                color = 'green' if is_correct else 'red'

                axes[images_shown].imshow(img)
                axes[images_shown].set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontweight='bold', fontsize=9)
                axes[images_shown].axis("off")

                images_shown += 1
                if images_shown >= n:
                    plt.tight_layout()
                    plt.show()
                    return

visualize_predictions(model, test_loader, n=6)


---
## 8. Model Serialization & Export
Save the model weights along with the class-to-index mapping and class names dictionary for deployment in the Streamlit application or REST API.


In [ ]:
export_data = {
    'model_state_dict': model.state_dict(),
    'class_to_idx': train_dataset.class_to_idx,
    'class_names': train_dataset.classes,
    'architecture': 'resnet18',
    'num_classes': len(train_dataset.classes)
}

os.makedirs('models', exist_ok=True)
export_path = 'models/crop_resnet18_exported.pth'
torch.save(export_data, export_path)

# Also save class names as standalone JSON
with open('models/class_names.json', 'w') as f:
    json.dump({
        'classes': train_dataset.classes,
        'class_to_idx': train_dataset.class_to_idx,
        'idx_to_class': {v: k for k, v in train_dataset.class_to_idx.items()}
    }, f, indent=2)

print(f"Model and class mapping exported successfully to {export_path} and models/class_names.json")
